# Retrieval Performance Evaluation: BiomedCLIP, BiomedCLIP+MLP, KG, and GNN

This notebook evaluates retrieval performance only. It does not generate reports.

Retrieval methods:

- `biomedclip_image_embedding`: BiomedCLIP image embedding retrieval.
- `biomedclip_zero_shot_label_vector`: BiomedCLIP zero-shot 14-label probability vector retrieval.
- `biomedclip_mlp_label_vector`: supervised BiomedCLIP+MLP 14-logit label-space retrieval.
- `kg_radgraph_entity_oracle`: KG entity retrieval using RadGraph observation/anatomy entities. This is an oracle semantic upper bound because query report entities are used.
- `gnn_*_label_vector`: GNN/KG prediction-vector retrieval if saved GNN prediction files are available.

Top-k values are `1, 5, 10, 50, 100, 200`. Metrics include MRR@k, Precision@k, Recall@k, mAP@k, nDCG@k, and Hit@k. nDCG@k is especially important here because it penalizes relevant cases placed lower in the retrieved list, which matches the clinical goal of prioritizing diagnostically useful evidence near the top of the context.

## 0. Optional Dependencies

In [ ]:
# Run on the cloud only if dependencies are missing.
# ! pip install -U open_clip_torch transformers faiss-cpu scikit-learn scipy nltk rouge-score bert-score pycocoevalcap radgraph

## 1. Imports and Configuration

In [1]:
import json
import math
import os
import random
import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Set, Tuple

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfTransformer

try:
    import faiss
except ImportError as exc:
    raise ImportError("Install faiss-cpu or faiss-gpu before running retrieval evaluation.") from exc

try:
    from scipy import sparse, stats
except Exception:
    sparse = None
    stats = None

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

DATA_ROOT = Path("/data/liangz2/openi/multi_kg") if Path("/data/liangz2/openi/multi_kg").exists() else Path.cwd()
OUTPUT_ROOT = DATA_ROOT / "retrieval_report_generation_eval"
EMBEDDING_DIR = OUTPUT_ROOT / "embeddings"
RETRIEVAL_RESULT_DIR = OUTPUT_ROOT / "retrieval_results"
for p in [OUTPUT_ROOT, EMBEDDING_DIR, RETRIEVAL_RESULT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SPLIT_CSV_CANDIDATES = [
    DATA_ROOT / "multi_kg_image_json_10fold.csv",
    Path("/data/liangz2/openi/inverse_diff/multi_kg_image_json_10fold.csv"),
    Path("/Users/liangz2/Documents/inverse_diffusion/multi_kg_image_json_10fold.csv"),
]

PREBUILT_FAISS_ROOTS = [
    Path("/data/liangz2/openi/inverse_diff/faiss_biomedclip"),
    DATA_ROOT / "faiss_biomedclip",
]
PREBUILT_INDEX_NAME = "multi_kg_train_test_biomedclip"

BIOMEDCLIP_BASELINE_ROOT = DATA_ROOT / "biomedclip_10fold_baselines"
KG_GRAPH_TABLE_DIR = DATA_ROOT / "kg_graph_tables"
KG_10FOLD_DIR = DATA_ROOT / "kg_10fold"
NON_TRANSFORMER_GNN_ROOT = DATA_ROOT / "kg_non_transformer_gnn_10fold"
TRANSFORMER_GNN_ROOT = DATA_ROOT / "kg_transformer_gnn_10fold"

BIOMEDCLIP_REPO = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

FOLD_INDICES = list(range(10))
K_VALUES = [1, 5, 10, 50, 100, 200]
MAX_K = max(K_VALUES)
MAX_FOLDS = None
MAX_TEST_QUERIES = None
SKIP_COMPLETED_RETRIEVAL = True

# GNN prediction-vector retrieval: query vector is saved GNN probabilities/logits;
# gallery vector is the observed training label vector. Change these to compare other GNN models.
GNN_RETRIEVAL_EXPERIMENTS = [
    {"family": "non_transformer_gnn", "model_name": "graphsage", "top_k": 500},
    {"family": "transformer_gnn", "model_name": "transformerconv", "top_k": 100},
]

BATCH_SIZE = 64
TEXT_BATCH_SIZE = 128
NUM_WORKERS = 0

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DEVICE:", DEVICE)

DATA_ROOT: /data/liangz2/openi/multi_kg
OUTPUT_ROOT: /data/liangz2/openi/multi_kg/retrieval_report_generation_eval
DEVICE: cuda


## 2. Label Schema and Split Table

In [2]:
CHEXPERT_LABELS = (
    "enlarged cardiomediastinum", "cardiomegaly", "lung opacity", "lung lesion", "edema",
    "consolidation", "pneumonia", "atelectasis", "pneumothorax",
    "pleural effusion", "pleural other", "fracture", "support devices",
)
SCORING_LABELS = ("normal",) + CHEXPERT_LABELS
TARGET_COLUMNS = ["y_" + label.replace(" ", "_") for label in SCORING_LABELS]

LABEL_ALIASES = {
    "enlarged cardiomediastinum": "enlarged cardiomediastinum",
    "cardiomediastinum": "enlarged cardiomediastinum",
    "cardiomegaly": "cardiomegaly",
    "lung opacity": "lung opacity",
    "opacity": "lung opacity",
    "lung lesion": "lung lesion",
    "lesion": "lung lesion",
    "edema": "edema",
    "pulmonary edema": "edema",
    "consolidation": "consolidation",
    "pneumonia": "pneumonia",
    "atelectasis": "atelectasis",
    "pneumothorax": "pneumothorax",
    "pleural effusion": "pleural effusion",
    "effusion": "pleural effusion",
    "pleural other": "pleural other",
    "fracture": "fracture",
    "support devices": "support devices",
    "support device": "support devices",
}

def find_first_existing(paths: Sequence[Path]) -> Optional[Path]:
    return next((p for p in paths if p.exists()), None)

def resolve_existing_path(path_value, suffix: Optional[str] = None) -> str:
    if path_value is None or str(path_value).lower() == "nan":
        return ""
    path = Path(str(path_value))
    if suffix:
        path = path.with_suffix(suffix)
    candidates = [path]
    text = str(path)
    replacements = [
        ("/vf/users/liangz2/openi/multi_kg", "/data/liangz2/openi/multi_kg"),
        ("/data/liangz2/openi/multi_kg", "/vf/users/liangz2/openi/multi_kg"),
    ]
    for src, dst in replacements:
        if text.startswith(src):
            candidates.append(Path(text.replace(src, dst, 1)))
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return str(path)

def parse_label_list(value) -> List[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, list):
        raw = value
    else:
        text = str(value).strip()
        if not text or text.lower() == "nan":
            return []
        try:
            raw = json.loads(text)
        except Exception:
            raw = re.split(r"[,;|]", text)
    labels = []
    for item in raw:
        key = re.sub(r"\\s+", " ", str(item).strip().lower())
        mapped = LABEL_ALIASES.get(key)
        if mapped and mapped not in labels:
            labels.append(mapped)
    return labels

def load_json_payload(path_value) -> Dict:
    path_text = resolve_existing_path(path_value)
    if not path_text:
        return {}
    path = Path(path_text)
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text())
    except Exception:
        return {}

def report_text_from_payload(payload: Dict) -> str:
    parts = []
    for field in ("INDICATION", "FINDINGS", "IMPRESSION"):
        text = str(payload.get(field, "") or "").strip()
        if text:
            parts.append(f"{field.title()}: {text}")
    if parts:
        return "\\n".join(parts)
    for field in ("Original_Caption", "Caption", "report", "description"):
        text = str(payload.get(field, "") or "").strip()
        if text:
            return text
    return ""

def ensure_targets(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    for col in TARGET_COLUMNS:
        if col not in frame.columns:
            frame[col] = 0
    if "labels_normalized" not in frame.columns:
        frame["labels_normalized"] = "[]"
    for i, row in frame.iterrows():
        if all(int(row.get(col, 0) or 0) == 0 for col in TARGET_COLUMNS):
            labels = parse_label_list(row.get("labels_normalized", "[]"))
            is_normal = str(row.get("normal", "")).strip().lower() == "yes" or not labels
            if is_normal:
                frame.at[i, "y_normal"] = 1
            for label in labels:
                col = "y_" + label.replace(" ", "_")
                if col in frame.columns:
                    frame.at[i, col] = 1
    for col in TARGET_COLUMNS:
        frame[col] = frame[col].fillna(0).astype(int)
    return frame

def load_split_table() -> pd.DataFrame:
    split_csv = find_first_existing(SPLIT_CSV_CANDIDATES)
    if split_csv is None:
        raise FileNotFoundError(f"Could not find split CSV. Checked: {SPLIT_CSV_CANDIDATES}")
    frame = pd.read_csv(split_csv).copy()
    frame["row_id"] = np.arange(len(frame), dtype=np.int64)
    frame["study_id"] = frame.get("study_id", frame["image_path"].map(lambda p: Path(str(p)).stem)).astype(str)
    frame["image_path"] = frame["image_path"].map(resolve_existing_path)
    if "json_path" not in frame.columns:
        frame["json_path"] = frame["image_path"].map(lambda p: str(Path(str(p)).with_suffix(".json")))
    frame["json_path"] = frame["json_path"].map(resolve_existing_path)
    frame["image_basename"] = frame["image_path"].map(lambda p: Path(str(p)).name)
    frame = ensure_targets(frame)
    if "report_text" not in frame.columns:
        reports = []
        for p in tqdm(frame["json_path"], desc="Load reference report text"):
            reports.append(report_text_from_payload(load_json_payload(p)))
        frame["report_text"] = reports
    return frame

split_df = load_split_table()
y_all = split_df[TARGET_COLUMNS].to_numpy(dtype=np.int8)
print("split rows", len(split_df))
display(split_df[["row_id", "study_id", "image_path", "labels_normalized"]].head())

Load reference report text:   0%|          | 0/44100 [00:00<?, ?it/s]

split rows 44100


,row_id,study_id,image_path,labels_normalized
0,0,50003542,/vf/users/liangz2/openi/multi_kg/train/5000354...,[]
1,1,50003755,/vf/users/liangz2/openi/multi_kg/train/5000375...,[]
2,2,50008255,/vf/users/liangz2/openi/multi_kg/train/5000825...,[]
3,3,50010166,/vf/users/liangz2/openi/multi_kg/train/5001016...,"[""lung opacity"", ""pneumonia""]"
4,4,50010229,/vf/users/liangz2/openi/multi_kg/train/5001022...,"[""atelectasis"", ""cardiomegaly"", ""consolidation..."


## 3. BiomedCLIP Image Embeddings and Zero-Shot Label Vectors

In [3]:
biomedclip_model = None
biomedclip_preprocess = None
biomedclip_tokenizer = None

def l2_normalize_np(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return (x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)).astype("float32")

def get_biomedclip_model():
    global biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer
    if biomedclip_model is None:
        from open_clip import create_model_from_pretrained, get_tokenizer
        model, preprocess = create_model_from_pretrained(BIOMEDCLIP_REPO)
        tokenizer = get_tokenizer(BIOMEDCLIP_REPO)
        model = model.to(DEVICE).eval()
        for p in model.parameters():
            p.requires_grad_(False)
        biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer = model, preprocess, tokenizer
    return biomedclip_model, biomedclip_preprocess, biomedclip_tokenizer

class ImagePathDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, preprocess):
        self.frame = frame.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        image_path = self.frame.loc[idx, "image_path"]
        with Image.open(image_path) as image:
            return self.preprocess(image.convert("RGB")), idx

@torch.inference_mode()
def encode_images(frame: pd.DataFrame) -> np.ndarray:
    model, preprocess, _ = get_biomedclip_model()
    loader = DataLoader(ImagePathDataset(frame, preprocess), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    chunks = []
    for images, _ in tqdm(loader, desc="BiomedCLIP image embeddings"):
        images = images.to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            feats = model.encode_image(images)
        chunks.append(F.normalize(feats.float(), dim=-1).cpu().numpy().astype("float32"))
    return np.concatenate(chunks, axis=0)

@torch.inference_mode()
def encode_texts(texts: Sequence[str]) -> np.ndarray:
    model, _, tokenizer = get_biomedclip_model()
    chunks = []
    for start in tqdm(range(0, len(texts), TEXT_BATCH_SIZE), desc="BiomedCLIP text embeddings"):
        tokens = tokenizer(list(texts[start:start + TEXT_BATCH_SIZE]), context_length=256).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
            feats = model.encode_text(tokens)
        chunks.append(F.normalize(feats.float(), dim=-1).cpu().numpy().astype("float32"))
    return np.concatenate(chunks, axis=0)

def cache_matches(meta_path: Path, frame: pd.DataFrame) -> bool:
    if not meta_path.exists():
        return False
    try:
        meta = pd.read_csv(meta_path)
    except Exception:
        return False
    return len(meta) == len(frame) and "row_id" in meta.columns and meta["row_id"].astype(int).tolist() == frame["row_id"].astype(int).tolist()

def load_prebuilt_image_embeddings(frame: pd.DataFrame) -> Optional[np.ndarray]:
    for root in PREBUILT_FAISS_ROOTS:
        emb_path = root / f"{PREBUILT_INDEX_NAME}_embeddings.npy"
        meta_path = root / f"{PREBUILT_INDEX_NAME}_metadata.csv"
        if not emb_path.exists() or not meta_path.exists():
            continue
        meta = pd.read_csv(meta_path)
        emb = np.load(emb_path).astype("float32")
        if "faiss_id" not in meta.columns:
            meta["faiss_id"] = np.arange(len(meta), dtype=int)
        if "image_basename" not in meta.columns:
            meta["image_basename"] = meta["image_path"].map(lambda p: Path(str(p)).name)
        meta["study_id"] = meta["study_id"].astype(str)
        lookup = meta.drop_duplicates(["study_id", "image_basename"]).set_index(["study_id", "image_basename"])["faiss_id"].to_dict()
        ids = []
        for _, row in frame.iterrows():
            ids.append(lookup.get((str(row["study_id"]), str(row["image_basename"])), -1))
        ids = np.asarray(ids, dtype=int)
        if np.all(ids >= 0):
            print("Loaded prebuilt image embeddings:", emb_path)
            return l2_normalize_np(emb[ids])
    return None

def get_image_embeddings(frame: pd.DataFrame) -> np.ndarray:
    emb_path = EMBEDDING_DIR / "biomedclip_image_embeddings.npy"
    meta_path = EMBEDDING_DIR / "biomedclip_image_embedding_metadata.csv"
    if emb_path.exists() and cache_matches(meta_path, frame):
        return np.load(emb_path).astype("float32")
    emb = load_prebuilt_image_embeddings(frame)
    if emb is None:
        emb = encode_images(frame)
    np.save(emb_path, emb.astype("float32"))
    frame[["row_id", "study_id", "image_basename", "image_path"]].to_csv(meta_path, index=False)
    return emb.astype("float32")

def positive_prompts(label: str) -> List[str]:
    if label == "normal":
        return ["a normal chest x-ray.", "a chest radiograph with no acute cardiopulmonary abnormality."]
    return [f"a chest x-ray showing {label}.", f"a chest radiograph with evidence of {label}."]

def negative_prompts(label: str) -> List[str]:
    if label == "normal":
        return ["an abnormal chest x-ray.", "a chest radiograph with cardiopulmonary abnormality."]
    return [f"a chest x-ray without {label}.", f"no radiographic evidence of {label}."]

def build_zero_shot_vectors(image_emb: np.ndarray) -> np.ndarray:
    cache = EMBEDDING_DIR / "biomedclip_zero_shot_label_probs.npy"
    if cache.exists():
        arr = np.load(cache).astype("float32")
        if arr.shape == (len(image_emb), len(SCORING_LABELS)):
            return arr
    pos_text, neg_text = [], []
    for label in SCORING_LABELS:
        pos_text.append(l2_normalize_np(encode_texts(positive_prompts(label))).mean(axis=0))
        neg_text.append(l2_normalize_np(encode_texts(negative_prompts(label))).mean(axis=0))
    pos_text = l2_normalize_np(np.stack(pos_text))
    neg_text = l2_normalize_np(np.stack(neg_text))
    model, _, _ = get_biomedclip_model()
    scale = float(model.logit_scale.exp().detach().cpu()) if hasattr(model, "logit_scale") else 100.0
    pos_logits = scale * (image_emb @ pos_text.T)
    neg_logits = scale * (image_emb @ neg_text.T)
    pair = np.stack([neg_logits, pos_logits], axis=-1)
    pair = pair - pair.max(axis=-1, keepdims=True)
    exp_pair = np.exp(pair)
    probs = exp_pair[..., 1] / exp_pair.sum(axis=-1)
    np.save(cache, probs.astype("float32"))
    return probs.astype("float32")

image_embeddings = get_image_embeddings(split_df)
zero_shot_vectors = build_zero_shot_vectors(image_embeddings)
print("image_embeddings", image_embeddings.shape, "zero_shot_vectors", zero_shot_vectors.shape)

image_embeddings (44100, 512) zero_shot_vectors (44100, 14)


## 4. BiomedCLIP + MLP Label Vectors

In [4]:
class BiomedClipMLPHead(nn.Module):
    def __init__(self, input_dim: int, num_outputs: int, hidden_dims: Sequence[int] = (512,), dropout: float = 0.20):
        super().__init__()
        layers = []
        prev = input_dim
        for hidden in hidden_dims:
            layers.extend([nn.Linear(prev, int(hidden)), nn.ReLU(inplace=True), nn.Dropout(dropout)])
            prev = int(hidden)
        layers.append(nn.Linear(prev, num_outputs))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.float())

def load_mlp_head_for_fold(fold: int, input_dim: int) -> Optional[nn.Module]:
    model_path = BIOMEDCLIP_BASELINE_ROOT / f"fold{fold}" / "models" / f"biomedclip_mlp_14_fold{fold}_seed42.pt"
    if not model_path.exists():
        print("Missing MLP checkpoint:", model_path)
        return None
    try:
        ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    except TypeError:
        ckpt = torch.load(model_path, map_location=DEVICE)
    hidden_dims = tuple(ckpt.get("hidden_dims", [512]))
    dropout = float(ckpt.get("dropout", 0.20))
    head = BiomedClipMLPHead(input_dim, len(SCORING_LABELS), hidden_dims=hidden_dims, dropout=dropout).to(DEVICE)
    state = ckpt.get("state_dict", ckpt)
    head.load_state_dict(state)
    head.eval()
    return head

@torch.inference_mode()
def predict_mlp_vectors(head: nn.Module, emb: np.ndarray, batch_size: int = 8192, output: str = "logits") -> np.ndarray:
    chunks = []
    for start in range(0, len(emb), batch_size):
        x = torch.from_numpy(emb[start:start + batch_size].astype("float32")).to(DEVICE)
        logits = head(x).detach().cpu().numpy().astype("float32")
        chunks.append(logits)
    logits = np.concatenate(chunks, axis=0) if chunks else np.empty((0, len(SCORING_LABELS)), dtype="float32")
    if output == "probs":
        return (1.0 / (1.0 + np.exp(-logits))).astype("float32")
    return logits.astype("float32")

def fold_mlp_vectors(fold: int, row_ids: np.ndarray) -> Optional[np.ndarray]:
    cache = EMBEDDING_DIR / f"fold{fold}_biomedclip_mlp_logits.npy"
    meta = EMBEDDING_DIR / f"fold{fold}_biomedclip_mlp_logits_meta.csv"
    if cache.exists() and meta.exists():
        meta_df = pd.read_csv(meta)
        if meta_df["row_id"].astype(int).tolist() == row_ids.astype(int).tolist():
            return np.load(cache).astype("float32")
    head = load_mlp_head_for_fold(fold, image_embeddings.shape[1])
    if head is None:
        return None
    logits = predict_mlp_vectors(head, image_embeddings[row_ids], output="logits")
    np.save(cache, logits.astype("float32"))
    pd.DataFrame({"row_id": row_ids.astype(int)}).to_csv(meta, index=False)
    return logits

## 5. KG Entity Sets from RadGraph

In [5]:
def load_kg_entity_sets(frame: pd.DataFrame) -> Dict[str, Set[str]]:
    edge_path = KG_GRAPH_TABLE_DIR / "kg_edges_radgraph_and_labels.csv"
    if not edge_path.exists():
        print("KG RadGraph edge table not found:", edge_path)
        return {str(sid): set() for sid in frame["study_id"].astype(str)}
    edges = pd.read_csv(edge_path)
    keep = edges["edge_type"].isin(["mentions", "uncertain_mentions", "located_at", "suggestive_of", "modify"])
    edges = edges[keep].copy()
    study_entities: Dict[str, Set[str]] = defaultdict(set)
    for _, row in edges.iterrows():
        sid = str(row.get("study_id", ""))
        etype = str(row.get("edge_type", ""))
        if not sid:
            continue
        if etype in {"mentions", "uncertain_mentions"}:
            dst = str(row.get("dst", ""))
            if dst:
                study_entities[sid].add(dst)
        elif etype in {"located_at", "suggestive_of", "modify"}:
            src, dst = str(row.get("src", "")), str(row.get("dst", ""))
            if src:
                study_entities[sid].add(src)
            if dst:
                study_entities[sid].add(dst)
    return {str(sid): study_entities.get(str(sid), set()) for sid in frame["study_id"].astype(str)}

kg_entity_sets_by_study = load_kg_entity_sets(split_df)
split_df["kg_entity_count"] = split_df["study_id"].astype(str).map(lambda sid: len(kg_entity_sets_by_study.get(sid, set())))
print("KG entity coverage:", float((split_df["kg_entity_count"] > 0).mean()))
display(split_df[["study_id", "kg_entity_count"]].head())

def entity_sets_for_rows(row_ids: np.ndarray) -> List[Set[str]]:
    return [kg_entity_sets_by_study.get(str(split_df.iloc[int(r)]["study_id"]), set()) for r in row_ids]

def build_entity_sparse_matrix(entity_sets: Sequence[Set[str]]):
    mlb = MultiLabelBinarizer(sparse_output=True)
    binary = mlb.fit_transform([sorted(s) for s in entity_sets])
    tfidf = TfidfTransformer(norm="l2", use_idf=True)
    matrix = tfidf.fit_transform(binary)
    return matrix, mlb, tfidf

def transform_entity_sparse_matrix(entity_sets: Sequence[Set[str]], mlb: MultiLabelBinarizer, tfidf: TfidfTransformer):
    binary = mlb.transform([sorted(s) for s in entity_sets])
    return tfidf.transform(binary)

KG entity coverage: 0.9777551020408163


,study_id,kg_entity_count
0,50003542,8
1,50003755,2
2,50008255,9
3,50010166,20
4,50010229,13


## 6. Retrieval Metrics

In [6]:
def fold_row_ids(fold: int) -> Tuple[np.ndarray, np.ndarray]:
    col = f"fold_{fold}"
    values = split_df[col].astype(str).str.lower()
    train_ids = np.flatnonzero(values.eq("train").to_numpy()).astype(np.int64)
    test_ids = np.flatnonzero(values.eq("test").to_numpy()).astype(np.int64)
    if MAX_TEST_QUERIES is not None:
        test_ids = test_ids[: int(MAX_TEST_QUERIES)]
    return train_ids, test_ids

def dense_search(train_vec: np.ndarray, test_vec: np.ndarray, k: int = MAX_K) -> Tuple[np.ndarray, np.ndarray]:
    train_vec = l2_normalize_np(train_vec.astype("float32"))
    test_vec = l2_normalize_np(test_vec.astype("float32"))
    index = faiss.IndexFlatIP(train_vec.shape[1])
    index.add(np.ascontiguousarray(train_vec))
    sims, idx = index.search(np.ascontiguousarray(test_vec), min(k, len(train_vec)))
    return sims.astype("float32"), idx.astype(np.int64)

def sparse_search(train_matrix, test_matrix, k: int = MAX_K) -> Tuple[np.ndarray, np.ndarray]:
    nn = NearestNeighbors(n_neighbors=min(k, train_matrix.shape[0]), metric="cosine", algorithm="brute")
    nn.fit(train_matrix)
    distances, idx = nn.kneighbors(test_matrix, return_distance=True)
    sims = 1.0 - distances
    return sims.astype("float32"), idx.astype(np.int64)

def relevant_indices_for_query(q_y: np.ndarray, train_y: np.ndarray, q_entities: Set[str], train_entity_sets: List[Set[str]], relevance: str) -> Set[int]:
    relevant: Set[int] = set()
    if relevance in {"label", "combined"}:
        q_disease = np.flatnonzero(q_y[1:] > 0) + 1
        if len(q_disease):
            hits = np.flatnonzero(train_y[:, q_disease].sum(axis=1) > 0)
        else:
            hits = np.flatnonzero(train_y[:, 0] > 0)
        relevant.update(map(int, hits))
    if relevance in {"entity", "combined"} and q_entities:
        for i, ents in enumerate(train_entity_sets):
            if q_entities.intersection(ents):
                relevant.add(int(i))
    return relevant

def apk(relevant_flags: np.ndarray, total_relevant: int) -> float:
    if total_relevant <= 0:
        return np.nan
    flags = relevant_flags.astype(bool)
    if flags.sum() == 0:
        return 0.0
    precision = np.cumsum(flags) / np.arange(1, len(flags) + 1)
    return float((precision * flags).sum() / min(total_relevant, len(flags)))

def mrr(relevant_flags: np.ndarray) -> float:
    hits = np.flatnonzero(relevant_flags.astype(bool))
    return 0.0 if len(hits) == 0 else float(1.0 / (hits[0] + 1))

def ndcg(relevant_flags: np.ndarray, total_relevant: int) -> float:
    flags = relevant_flags.astype(float)
    if total_relevant <= 0:
        return np.nan
    discounts = 1.0 / np.log2(np.arange(2, len(flags) + 2))
    dcg = float((flags * discounts).sum())
    ideal_len = min(total_relevant, len(flags))
    ideal = float(discounts[:ideal_len].sum())
    return dcg / ideal if ideal > 0 else np.nan

def evaluate_neighbors(
    train_ids: np.ndarray,
    test_ids: np.ndarray,
    sims: np.ndarray,
    idx: np.ndarray,
    method_name: str,
    fold: int,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_y = y_all[train_ids]
    test_y = y_all[test_ids]
    train_entities = entity_sets_for_rows(train_ids)
    test_entities = entity_sets_for_rows(test_ids)

    summary_rows = []
    query_rows = []
    for relevance in ["label", "entity", "combined"]:
        for k in K_VALUES:
            values = defaultdict(list)
            for qi in range(len(test_ids)):
                nn_idx = idx[qi, :k]
                rel = relevant_indices_for_query(test_y[qi], train_y, test_entities[qi], train_entities, relevance)
                flags = np.array([int(int(j) in rel) for j in nn_idx if int(j) >= 0], dtype=int)
                total_rel = len(rel)
                precision = float(flags.mean()) if len(flags) else np.nan
                recall = float(flags.sum() / total_rel) if total_rel else np.nan
                row = {
                    "fold": fold,
                    "retrieval_method": method_name,
                    "relevance": relevance,
                    "k": int(k),
                    "query_row_id": int(test_ids[qi]),
                    "num_relevant_gallery": int(total_rel),
                    "precision_at_k": precision,
                    "recall_at_k": recall,
                    "ap_at_k": apk(flags, total_rel),
                    "mrr_at_k": mrr(flags),
                    "ndcg_at_k": ndcg(flags, total_rel),
                    "hit_at_k": float(flags.sum() > 0) if len(flags) else np.nan,
                }
                query_rows.append(row)
                for metric in ["precision_at_k", "recall_at_k", "ap_at_k", "mrr_at_k", "ndcg_at_k", "hit_at_k"]:
                    values[metric].append(row[metric])
            summary = {
                "fold": fold,
                "retrieval_method": method_name,
                "relevance": relevance,
                "k": int(k),
                "num_queries": int(len(test_ids)),
            }
            for metric, vals in values.items():
                summary[metric] = float(np.nanmean(vals))
            summary_rows.append(summary)
    return pd.DataFrame(summary_rows), pd.DataFrame(query_rows)

def save_neighbor_audit(method_dir: Path, fold: int, method_name: str, train_ids: np.ndarray, test_ids: np.ndarray, sims: np.ndarray, idx: np.ndarray, max_queries: int = 25):
    rows = []
    for qi in range(min(max_queries, len(test_ids))):
        q = split_df.iloc[int(test_ids[qi])]
        for rank, j in enumerate(idx[qi, :min(MAX_K, idx.shape[1])], start=1):
            if j < 0:
                continue
            n = split_df.iloc[int(train_ids[int(j)])]
            rows.append({
                "fold": fold,
                "retrieval_method": method_name,
                "query_row_id": int(test_ids[qi]),
                "query_study_id": q["study_id"],
                "query_labels": q.get("labels_normalized", ""),
                "rank": rank,
                "similarity": float(sims[qi, rank - 1]),
                "neighbor_row_id": int(train_ids[int(j)]),
                "neighbor_study_id": n["study_id"],
                "neighbor_labels": n.get("labels_normalized", ""),
                "neighbor_report": str(n.get("report_text", ""))[:800],
            })
    pd.DataFrame(rows).to_csv(method_dir / f"{method_name}_fold{fold}_neighbor_audit.csv", index=False)

## 7. Retrieval Vector Builders

In [7]:
def gnn_prediction_path(fold: int, family: str, model_name: str, top_k: int) -> Optional[Path]:
    root = TRANSFORMER_GNN_ROOT if family == "transformer_gnn" else NON_TRANSFORMER_GNN_ROOT
    candidates = [
        root / model_name / f"top_k_{top_k}" / f"fold{fold}" / "predictions" / f"{model_name}_test_predictions.csv",
        root / model_name / f"top_k_{top_k}" / f"fold{fold}" / "predictions" / "graphsage_test_predictions.csv",
    ]
    return next((p for p in candidates if p.exists()), None)

def load_gnn_test_vectors(fold: int, family: str, model_name: str, top_k: int, test_ids: np.ndarray) -> Optional[np.ndarray]:
    path = gnn_prediction_path(fold, family, model_name, top_k)
    if path is None:
        print("Missing GNN prediction file:", family, model_name, top_k, "fold", fold)
        return None
    pred = pd.read_csv(path)
    prob_cols = [f"prob_{label}" for label in SCORING_LABELS]
    if not set(prob_cols).issubset(pred.columns):
        print("GNN prediction file missing probability columns:", path)
        return None
    if "study_id" in pred.columns:
        pred["study_id"] = pred["study_id"].astype(str)
        lookup = pred.drop_duplicates("study_id").set_index("study_id")
        rows = []
        for rid in test_ids:
            sid = str(split_df.iloc[int(rid)]["study_id"])
            if sid not in lookup.index:
                return None
            rows.append(lookup.loc[sid, prob_cols].to_numpy(dtype=np.float32))
        return np.stack(rows).astype("float32")
    return pred[prob_cols].to_numpy(dtype=np.float32)[: len(test_ids)]

def build_retrieval_candidates_for_fold(fold: int) -> List[Dict]:
    train_ids, test_ids = fold_row_ids(fold)
    out = []

    out.append({
        "name": "biomedclip_image_embedding",
        "type": "dense",
        "train_vec": image_embeddings[train_ids],
        "test_vec": image_embeddings[test_ids],
        "description": "BiomedCLIP image embedding cosine retrieval.",
    })
    out.append({
        "name": "biomedclip_zero_shot_label_vector",
        "type": "dense",
        "train_vec": zero_shot_vectors[train_ids],
        "test_vec": zero_shot_vectors[test_ids],
        "description": "BiomedCLIP zero-shot 14-label probability-vector retrieval.",
    })

    all_ids_for_fold = np.concatenate([train_ids, test_ids])
    mlp_all = fold_mlp_vectors(fold, all_ids_for_fold)
    if mlp_all is not None:
        out.append({
            "name": "biomedclip_mlp_label_vector",
            "type": "dense",
            "train_vec": mlp_all[: len(train_ids)],
            "test_vec": mlp_all[len(train_ids):],
            "description": "Supervised BiomedCLIP+MLP 14-logit label-space retrieval.",
        })

    # Oracle KG entity retrieval: uses true report-derived RadGraph entities for query and gallery.
    train_entity_sets = entity_sets_for_rows(train_ids)
    test_entity_sets = entity_sets_for_rows(test_ids)
    if any(len(s) for s in train_entity_sets) and any(len(s) for s in test_entity_sets):
        train_sparse, mlb, tfidf = build_entity_sparse_matrix(train_entity_sets)
        test_sparse = transform_entity_sparse_matrix(test_entity_sets, mlb, tfidf)
        out.append({
            "name": "kg_radgraph_entity_oracle",
            "type": "sparse",
            "train_vec": train_sparse,
            "test_vec": test_sparse,
            "description": "Oracle KG retrieval from RadGraph observation/anatomy entity TF-IDF vectors.",
        })

    # GNN/KG prediction-vector retrieval: query GNN probabilities vs observed training label vectors.
    for exp in GNN_RETRIEVAL_EXPERIMENTS:
        test_gnn = load_gnn_test_vectors(fold, exp["family"], exp["model_name"], int(exp["top_k"]), test_ids)
        if test_gnn is not None:
            out.append({
                "name": f"gnn_{exp['family']}_{exp['model_name']}_topk{int(exp['top_k'])}_label_vector",
                "type": "dense",
                "train_vec": y_all[train_ids].astype("float32"),
                "test_vec": test_gnn.astype("float32"),
                "description": "GNN/KG predicted 14-label vector retrieval against training label vectors.",
            })
    return out

## 8. Run Retrieval Evaluation

In [8]:
all_summary = []
all_query_metrics = []
folds_to_run = FOLD_INDICES[:MAX_FOLDS] if MAX_FOLDS is not None else FOLD_INDICES

for fold in folds_to_run:
    train_ids, test_ids = fold_row_ids(fold)
    candidates = build_retrieval_candidates_for_fold(fold)
    print(f"\\n===== fold={fold} train={len(train_ids)} test={len(test_ids)} methods={len(candidates)} =====")
    for cand in candidates:
        method = cand["name"]
        method_dir = RETRIEVAL_RESULT_DIR / method / f"fold{fold}"
        method_dir.mkdir(parents=True, exist_ok=True)
        complete = method_dir / "FOLD_COMPLETE.json"
        summary_path = method_dir / f"{method}_fold{fold}_retrieval_summary.csv"
        query_path = method_dir / f"{method}_fold{fold}_query_metrics.csv"
        if SKIP_COMPLETED_RETRIEVAL and complete.exists() and summary_path.exists():
            print("skip", method, "fold", fold)
            summary = pd.read_csv(summary_path)
            query_metrics = pd.read_csv(query_path) if query_path.exists() else pd.DataFrame()
        else:
            print("run", method, cand["description"])
            if cand["type"] == "sparse":
                sims, idx = sparse_search(cand["train_vec"], cand["test_vec"], MAX_K)
            else:
                sims, idx = dense_search(cand["train_vec"], cand["test_vec"], MAX_K)
            summary, query_metrics = evaluate_neighbors(train_ids, test_ids, sims, idx, method, fold)
            summary["description"] = cand["description"]
            summary.to_csv(summary_path, index=False)
            query_metrics.to_csv(query_path, index=False)
            save_neighbor_audit(method_dir, fold, method, train_ids, test_ids, sims, idx)
            complete.write_text(json.dumps({"fold": fold, "method": method, "status": "complete"}, indent=2))

        all_summary.append(summary)
        if len(query_metrics):
            all_query_metrics.append(query_metrics)
        pd.concat(all_summary, ignore_index=True).to_csv(OUTPUT_ROOT / "retrieval_10fold_summary_progress.csv", index=False)

retrieval_summary = pd.concat(all_summary, ignore_index=True)
retrieval_summary.to_csv(OUTPUT_ROOT / "retrieval_10fold_summary.csv", index=False)
if all_query_metrics:
    pd.concat(all_query_metrics, ignore_index=True).to_csv(OUTPUT_ROOT / "retrieval_10fold_query_metrics.csv", index=False)
display(retrieval_summary.head())

/data/liangz2/diffuser2/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['anatomy::1', 'anatomy::10th', 'anatomy::adrenal', 'anatomy::anteromedially', 'anatomy::aortopulmonic', 'anatomy::bibasalir', 'anatomy::cartilages', 'anatomy::compliance', 'anatomy::conduction', 'anatomy::currently', 'anatomy::cvc', 'anatomy::deserves', 'anatomy::diameters', 'anatomy::diaphgragm', 'anatomy::ending', 'anatomy::extending', 'anatomy::extrathoracic', 'anatomy::fracutre', 'anatomy::gross', 'anatomy::history', 'anatomy::hypertrophy', 'anatomy::indwelling', 'anatomy::l2 vertebral', 'anatomy::laterall', 'anatomy::look', 'anatomy::lucencies', 'anatomy::minimally', 'anatomy::monitor', 'anatomy::much', 'anatomy::overlap', 'anatomy::ower', 'anatomy::pancoast', 'anatomy::pelves', 'anatomy::periaortic', 'anatomy::pheresis', 'anatomy::pleurex', 'anatomy::reaction', 'anatomy::rhonchi', 'anatomy::sabersheath', 'anatomy::side has', 'anatomy::sided medial', 'anatomy:

\n===== fold=0 train=39690 test=4410 methods=6 =====
skip biomedclip_image_embedding fold 0
skip biomedclip_zero_shot_label_vector fold 0
skip biomedclip_mlp_label_vector fold 0
skip kg_radgraph_entity_oracle fold 0
skip gnn_non_transformer_gnn_graphsage_topk500_label_vector fold 0
skip gnn_transformer_gnn_transformerconv_topk100_label_vector fold 0


/data/liangz2/diffuser2/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['anatomy::antero', 'anatomy::arteriovenous', 'anatomy::assist', 'anatomy::atriuym', 'anatomy::based lass', 'anatomy::behind', 'anatomy::canthus', 'anatomy::chains', 'anatomy::confirmed', 'anatomy::degeneration', 'anatomy::described', 'anatomy::deviating', 'anatomy::dorsally', 'anatomy::earlier', 'anatomy::esophagogastric', 'anatomy::external', 'anatomy::fall', 'anatomy::form', 'anatomy::four', 'anatomy::heavy', 'anatomy::identified', 'anatomy::inflammatory', 'anatomy::inflated', 'anatomy::infradiaphragmatic', 'anatomy::issues', 'anatomy::juxtacardiac', 'anatomy::known', 'anatomy::leading', 'anatomy::levels along', 'anatomy::luminal', 'anatomy::lung that', 'anatomy::majority', 'anatomy::markedly', 'anatomy::measure', 'anatomy::most sternal', 'anatomy::motility', 'anatomy::old', 'anatomy::pe', 'anatomy::piriform', 'anatomy::plana', 'anatomy::plane', 'anatomy::plexopa

\n===== fold=1 train=39690 test=4410 methods=6 =====
skip biomedclip_image_embedding fold 1
skip biomedclip_zero_shot_label_vector fold 1
skip biomedclip_mlp_label_vector fold 1
skip kg_radgraph_entity_oracle fold 1
skip gnn_non_transformer_gnn_graphsage_topk500_label_vector fold 1
skip gnn_transformer_gnn_transformerconv_topk100_label_vector fold 1


/data/liangz2/diffuser2/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['anatomy::6 cm', 'anatomy::acute', 'anatomy::auricle', 'anatomy::bandlike', 'anatomy::basal segment', 'anatomy::cadiopulmonary', 'anatomy::caliber', 'anatomy::cardiacsize', 'anatomy::cardiomediastial', 'anatomy::cephalization', 'anatomy::contribute', 'anatomy::deviates', 'anatomy::distortion', 'anatomy::each', 'anatomy::endovascular', 'anatomy::hemidiagraphms', 'anatomy::her', 'anatomy::hips', 'anatomy::horacic', 'anatomy::humeri', 'anatomy::indicates', 'anatomy::inter', 'anatomy::intraabdominal', 'anatomy::intracranial', 'anatomy::iodine', 'anatomy::kerley', 'anatomy::lobulated', 'anatomy::lung disease', 'anatomy::maintains', 'anatomy::material', 'anatomy::mediastinl', 'anatomy::most pedicle', 'anatomy::osteopathy', 'anatomy::particular', 'anatomy::percutaneously', 'anatomy::prevertebral', 'anatomy::processed', 'anatomy::pulmononary', 'anatomy::pulnoary', 'anatomy

...

/data/liangz2/diffuser2/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['anatomy::10', 'anatomy::4 cm', 'anatomy::abdominal process', 'anatomy::accounts', 'anatomy::acromion', 'anatomy::aerated', 'anatomy::aligns', 'anatomy::anterolaterally', 'anatomy::apicolaterally', 'anatomy::asthma cardiac', 'anatomy::attenuation', 'anatomy::bibasialr', 'anatomy::cardiacmediastinal', 'anatomy::cement', 'anatomy::column e', 'anatomy::compressive', 'anatomy::decreased', 'anatomy::diaphragam', 'anatomy::displaced', 'anatomy::drug', 'anatomy::estimated', 'anatomy::extraparenchymal', 'anatomy::faint', 'anatomy::girdles', 'anatomy::having', 'anatomy::hematoma', 'anatomy::induced', 'anatomy::interstital', 'anatomy::intramuscular', 'anatomy::intravenous', 'anatomy::iv', 'anatomy::lateral arch', 'anatomy::likley', 'anatomy::ling', 'anatomy::lowe', 'anatomy::lower hemithorax', 'anatomy::mid esophagus', 'anatomy::mogul', 'anatomy::monitoring', 'anatomy::nodal

\n===== fold=8 train=39690 test=4410 methods=6 =====
skip biomedclip_image_embedding fold 8
skip biomedclip_zero_shot_label_vector fold 8
skip biomedclip_mlp_label_vector fold 8
skip kg_radgraph_entity_oracle fold 8
skip gnn_non_transformer_gnn_graphsage_topk500_label_vector fold 8
skip gnn_transformer_gnn_transformerconv_topk100_label_vector fold 8


/data/liangz2/diffuser2/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['anatomy::5 cm above', 'anatomy::aortic lines', 'anatomy::artifact', 'anatomy::being', 'anatomy::bleb', 'anatomy::brachycephalic', 'anatomy::branch', 'anatomy::bronchiovasculature', 'anatomy::bronchopleural', 'anatomy::bullous', 'anatomy::cap', 'anatomy::cardiomedistinal', 'anatomy::cardiopulmonry', 'anatomy::carotids', 'anatomy::cava', 'anatomy::centrally', 'anatomy::cm distal', 'anatomy::coils', 'anatomy::concerning', 'anatomy::coracoid', 'anatomy::corevalve', 'anatomy::coronaries', 'anatomy::cortex', 'anatomy::date', 'anatomy::dermal', 'anatomy::displacing', 'anatomy::effort', 'anatomy::electrodes', 'anatomy::equal', 'anatomy::evidence', 'anatomy::extent', 'anatomy::foot', 'anatomy::formation', 'anatomy::hd', 'anatomy::herniation', 'anatomy::hyper', 'anatomy::immediately', 'anatomy::infraclavicular', 'anatomy::infrahepatic', 'anatomy::intraparenchymal', 'anatomy

\n===== fold=9 train=39690 test=4410 methods=6 =====
skip biomedclip_image_embedding fold 9
skip biomedclip_zero_shot_label_vector fold 9
run biomedclip_mlp_label_vector Supervised BiomedCLIP+MLP 14-logit label-space retrieval.
run kg_radgraph_entity_oracle Oracle KG retrieval from RadGraph observation/anatomy entity TF-IDF vectors.
run gnn_non_transformer_gnn_graphsage_topk500_label_vector GNN/KG predicted 14-label vector retrieval against training label vectors.
run gnn_transformer_gnn_transformerconv_topk100_label_vector GNN/KG predicted 14-label vector retrieval against training label vectors.


,fold,retrieval_method,relevance,k,num_queries,precision_at_k,recall_at_k,ap_at_k,mrr_at_k,ndcg_at_k,hit_at_k,description
0,0,biomedclip_image_embedding,label,1,4410,0.661224,0.000046,0.661224,0.661224,0.661224,0.661224,BiomedCLIP image embedding cosine retrieval.
1,0,biomedclip_image_embedding,label,5,4410,0.650431,0.000221,0.582878,0.747252,0.652501,0.872789,BiomedCLIP image embedding cosine retrieval.
2,0,biomedclip_image_embedding,label,10,4410,0.644059,0.000421,0.556626,0.752497,0.647318,0.912018,BiomedCLIP image embedding cosine retrieval.
3,0,biomedclip_image_embedding,label,50,4410,0.635029,0.002025,0.519673,0.755903,0.638090,0.975283,BiomedCLIP image embedding cosine retrieval.
4,0,biomedclip_image_embedding,label,100,4410,0.630256,0.003952,0.509479,0.756090,0.633290,0.987755,BiomedCLIP image embedding cosine retrieval.


## 9. Aggregate Retrieval Results and Paired Tests

In [9]:
retrieval_summary = pd.read_csv(OUTPUT_ROOT / "retrieval_10fold_summary.csv")
metric_cols = ["precision_at_k", "recall_at_k", "ap_at_k", "mrr_at_k", "ndcg_at_k", "hit_at_k"]
aggregate = retrieval_summary.groupby(["retrieval_method", "relevance", "k"])[metric_cols].agg(["mean", "std"]).reset_index()
aggregate.columns = ["_".join(c).strip("_") for c in aggregate.columns]
aggregate = aggregate.sort_values(["relevance", "k", "mrr_at_k_mean"], ascending=[True, True, False])
aggregate.to_csv(OUTPUT_ROOT / "retrieval_10fold_aggregate_summary.csv", index=False)
display(aggregate)

def bh_fdr(p_values):
    p = np.asarray(p_values, dtype=float)
    q = np.full_like(p, np.nan)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return q
    idx = np.where(mask)[0]
    order = np.argsort(p[mask])
    sorted_idx = idx[order]
    sorted_p = p[sorted_idx]
    ranks = np.arange(1, len(sorted_p) + 1)
    sorted_q = sorted_p * len(sorted_p) / ranks
    sorted_q = np.minimum.accumulate(sorted_q[::-1])[::-1]
    q[sorted_idx] = np.clip(sorted_q, 0, 1)
    return q

baseline_name = "biomedclip_image_embedding"
rows = []
for (method, relevance, k), cand in retrieval_summary[retrieval_summary["retrieval_method"] != baseline_name].groupby(["retrieval_method", "relevance", "k"]):
    base = retrieval_summary[
        retrieval_summary["retrieval_method"].eq(baseline_name)
        & retrieval_summary["relevance"].eq(relevance)
        & retrieval_summary["k"].eq(k)
    ]
    merged = cand.merge(base, on=["fold", "relevance", "k"], suffixes=("_candidate", "_baseline"))
    if merged.empty:
        continue
    for metric in metric_cols:
        c = merged[f"{metric}_candidate"].to_numpy(dtype=float)
        b = merged[f"{metric}_baseline"].to_numpy(dtype=float)
        valid = np.isfinite(c) & np.isfinite(b)
        if valid.sum() < 2:
            continue
        diff = c[valid] - b[valid]
        p_t = np.nan
        p_w = np.nan
        if stats is not None and np.nanstd(diff, ddof=1) > 0:
            p_t = float(stats.ttest_rel(c[valid], b[valid]).pvalue)
            try:
                p_w = float(stats.wilcoxon(diff[np.abs(diff) > 1e-12]).pvalue)
            except Exception:
                p_w = np.nan
        rows.append({
            "retrieval_method": method,
            "baseline": baseline_name,
            "relevance": relevance,
            "k": int(k),
            "metric": metric,
            "candidate_mean": float(np.nanmean(c[valid])),
            "baseline_mean": float(np.nanmean(b[valid])),
            "mean_diff": float(np.nanmean(diff)),
            "win_folds": int((diff > 0).sum()),
            "loss_folds": int((diff < 0).sum()),
            "paired_t_p": p_t,
            "wilcoxon_p": p_w,
        })
pairwise = pd.DataFrame(rows)
if not pairwise.empty:
    pairwise["paired_t_q"] = bh_fdr(pairwise["paired_t_p"])
    pairwise["wilcoxon_q"] = bh_fdr(pairwise["wilcoxon_p"])
pairwise.to_csv(OUTPUT_ROOT / "retrieval_pairwise_tests_vs_biomedclip_image.csv", index=False)
display(pairwise.sort_values(["relevance", "k", "metric", "mean_diff"], ascending=[True, True, True, False]).head(80))

,retrieval_method,relevance,k,precision_at_k_mean,precision_at_k_std,recall_at_k_mean,recall_at_k_std,ap_at_k_mean,ap_at_k_std,mrr_at_k_mean,mrr_at_k_std,ndcg_at_k_mean,ndcg_at_k_std,hit_at_k_mean,hit_at_k_std
72,gnn_transformer_gnn_transformerconv_topk100_la...,combined,1,0.986213,0.003835,0.000026,2.945923e-07,0.986213,0.003835,0.986213,0.003835,0.986213,0.003835,0.986213,0.003835
54,gnn_non_transformer_gnn_graphsage_topk500_labe...,combined,1,0.982744,0.009023,0.000026,3.425679e-07,0.982744,0.009023,0.982744,0.009023,0.982744,0.009023,0.982744,0.009023
90,kg_radgraph_entity_oracle,combined,1,0.980363,0.006129,0.000026,3.102870e-07,0.980363,0.006129,0.980363,0.006129,0.980363,0.006129,0.980363,0.006129
0,biomedclip_image_embedding,combined,1,0.975896,0.001747,0.000026,1.566876e-07,0.975896,0.001747,0.975896,0.001747,0.975896,0.001747,0.975896,0.001747
18,biomedclip_mlp_label_vector,combined,1,0.975850,0.001852,0.000026,1.499496e-07,0.975850,0.001852,0.975850,0.001852,0.975850,0.001852,0.975850,0.001852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17,biomedclip_image_embedding,label,200,0.627279,0.001969,0.007753,6.484322e-05,0.502267,0.001982,0.751991,0.004099,0.629808,0.001954,0.996531,0.000659
35,biomedclip_mlp_label_vector,label,200,0.651491,0.002380,0.009017,6.699260e-05,0.530085,0.002667,0.751177,0.004777,0.651480,0.002400,0.994535,0.001192
53,biomedclip_zero_shot_label_vector,label,200,0.596465,0.002181,0.007796,8.982543e-05,0.454158,0.001863,0.723267,0.003486,0.597852,0.002159,0.996712,0.000711
89,gnn_transformer_gnn_transformerconv_topk100_la...,label,200,0.612382,0.162469,0.011611,1.123508e-03,0.580773,0.164450,0.622096,0.158606,0.612190,0.162264,0.698662,0.153666


,retrieval_method,baseline,relevance,k,metric,candidate_mean,baseline_mean,mean_diff,win_folds,loss_folds,paired_t_p,wilcoxon_p,paired_t_q,wilcoxon_q
326,gnn_transformer_gnn_transformerconv_topk100_la...,biomedclip_image_embedding,combined,1,ap_at_k,0.986213,0.975896,0.010317,10,0,7.004506e-05,0.001953,2.044558e-04,0.004469
218,gnn_non_transformer_gnn_graphsage_topk500_labe...,biomedclip_image_embedding,combined,1,ap_at_k,0.982744,0.975896,0.006848,9,1,4.053236e-02,0.083984,6.755394e-02,0.134176
434,kg_radgraph_entity_oracle,biomedclip_image_embedding,combined,1,ap_at_k,0.980363,0.975896,0.004467,8,1,7.175312e-02,0.019531,1.139608e-01,0.035274
2,biomedclip_mlp_label_vector,biomedclip_image_embedding,combined,1,ap_at_k,0.975850,0.975896,-0.000045,4,6,9.512660e-01,0.867188,9.723762e-01,0.911053
110,biomedclip_zero_shot_label_vector,biomedclip_image_embedding,combined,1,ap_at_k,0.973333,0.975896,-0.002562,2,7,3.938496e-02,0.054688,6.672546e-02,0.090587
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,kg_radgraph_entity_oracle,biomedclip_image_embedding,combined,10,ndcg_at_k,0.990819,0.975288,0.015531,10,0,1.373067e-08,0.001953,7.129386e-08,0.004469
232,gnn_non_transformer_gnn_graphsage_topk500_labe...,biomedclip_image_embedding,combined,10,ndcg_at_k,0.977381,0.975288,0.002093,6,4,3.198216e-01,0.375000,4.212285e-01,0.476471
340,gnn_transformer_gnn_transformerconv_topk100_la...,biomedclip_image_embedding,combined,10,ndcg_at_k,0.977053,0.975288,0.001765,7,3,7.170344e-01,0.556641,8.033166e-01,0.656301
16,biomedclip_mlp_label_vector,biomedclip_image_embedding,combined,10,ndcg_at_k,0.975637,0.975288,0.000349,6,4,2.372614e-01,0.193359,3.302092e-01,0.275499


## 10. nDCG-Ranked Retrieval Summary

In [10]:
aggregate_path = OUTPUT_ROOT / "retrieval_10fold_aggregate_summary.csv"
if aggregate_path.exists():
    aggregate_for_ndcg = pd.read_csv(aggregate_path)
    ndcg_ranked = aggregate_for_ndcg.sort_values(
        ["relevance", "k", "ndcg_at_k_mean", "mrr_at_k_mean"],
        ascending=[True, True, False, False],
    )
    ndcg_ranked_path = OUTPUT_ROOT / "retrieval_10fold_aggregate_summary_ranked_by_ndcg.csv"
    ndcg_ranked.to_csv(ndcg_ranked_path, index=False)
    display(ndcg_ranked.head(80))
    print("Saved", ndcg_ranked_path)
else:
    print("Run the aggregate-results cell first:", aggregate_path)

,retrieval_method,relevance,k,precision_at_k_mean,precision_at_k_std,recall_at_k_mean,recall_at_k_std,ap_at_k_mean,ap_at_k_std,mrr_at_k_mean,mrr_at_k_std,ndcg_at_k_mean,ndcg_at_k_std,hit_at_k_mean,hit_at_k_std
0,gnn_transformer_gnn_transformerconv_topk100_la...,combined,1,0.986213,0.003835,0.000026,2.945923e-07,0.986213,0.003835,0.986213,0.003835,0.986213,0.003835,0.986213,0.003835
1,gnn_non_transformer_gnn_graphsage_topk500_labe...,combined,1,0.982744,0.009023,0.000026,3.425679e-07,0.982744,0.009023,0.982744,0.009023,0.982744,0.009023,0.982744,0.009023
2,kg_radgraph_entity_oracle,combined,1,0.980363,0.006129,0.000026,3.102870e-07,0.980363,0.006129,0.980363,0.006129,0.980363,0.006129,0.980363,0.006129
3,biomedclip_image_embedding,combined,1,0.975896,0.001747,0.000026,1.566876e-07,0.975896,0.001747,0.975896,0.001747,0.975896,0.001747,0.975896,0.001747
4,biomedclip_mlp_label_vector,combined,1,0.975850,0.001852,0.000026,1.499496e-07,0.975850,0.001852,0.975850,0.001852,0.975850,0.001852,0.975850,0.001852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,biomedclip_zero_shot_label_vector,label,1,0.609320,0.003980,0.000041,9.656996e-07,0.609320,0.003980,0.609320,0.003980,0.609320,0.003980,0.609320,0.003980
76,gnn_non_transformer_gnn_graphsage_topk500_labe...,label,1,0.606485,0.167323,0.000055,1.234762e-05,0.606485,0.167323,0.606485,0.167323,0.606485,0.167323,0.606485,0.167323
77,gnn_transformer_gnn_transformerconv_topk100_la...,label,1,0.602857,0.157343,0.000069,7.334882e-06,0.602857,0.157343,0.602857,0.157343,0.602857,0.157343,0.602857,0.157343
78,kg_radgraph_entity_oracle,label,5,0.781438,0.004126,0.000349,9.085424e-06,0.735257,0.005734,0.847494,0.004689,0.782860,0.004408,0.935850,0.004863


Saved /data/liangz2/openi/multi_kg/retrieval_report_generation_eval/retrieval_10fold_aggregate_summary_ranked_by_ndcg.csv


## Retrieval-Only Outputs

The primary outputs are saved under `/data/liangz2/openi/multi_kg/retrieval_report_generation_eval`:

- `retrieval_10fold_summary.csv`
- `retrieval_10fold_aggregate_summary.csv`
- `retrieval_pairwise_tests_vs_biomedclip_image.csv`
- `retrieval_10fold_aggregate_summary_ranked_by_ndcg.csv`
- per-method/fold query metrics and neighbor audit files under `retrieval_results/`

Use these files to select the best retrieval method/k settings for the report-generation notebook.